In [1]:
!pip install -q sentence-transformers faiss-gpu-cu12 transformers datasets pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 18.8 MB/s eta 0:00:00


In [2]:
# @title 1. Environment Setup & Imports
import os
import gc
import json
import csv
import torch
import faiss
import ast
import random
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
from pathlib import Path
from google.colab import drive
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, AutoConfig
import torch.nn as nn
import torch.nn.functional as F

# Mount Google Drive
drive.mount('/content/drive')

# Set random seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Environment set up and Drive mounted.")

Mounted at /content/drive
Environment set up and Drive mounted.


In [3]:
print("test")
res = faiss.StandardGpuResources()

test


In [4]:
# @title 2. Configuration & Dynamic Logger
class Config:
    # --- Paths ---
    BASE_DIR = "/content/drive/MyDrive/RAGSearch"
    PRODUCT_PATH = f"{BASE_DIR}/preprocessed_products.parquet"
    QUERY_PATH = f"{BASE_DIR}/preprocessed_queries.parquet"

    # Models
    HEYDARI_MODEL = "heydariAI/persian-embeddings"
    HOOSHVARE_BASE = "HooshvareLab/distilbert-fa-zwnj-base"
    CUSTOM_BASE_PATH = f"{BASE_DIR}/base_model/"
    CUSTOM_FULL_PTH = f"{BASE_DIR}/retrieval_system_full.pth"

    # Logging
    LOG_FILE = f"{BASE_DIR}/experiment_logs.csv"

    # --- Settings ---
    BATCH_SIZE = 64
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_FAISS_GPU = True  # Toggle for FAISS GPU usage

    # --- Data Reduction (Issue 2) ---
    # Fraction of queries to use (0.0 to 1.0).
    # e.g., 0.1 means use 10% of data, keeping category distribution.
    QUERY_SUBSET_RATIO = 0.5

    # --- Evaluation Metrics (Issue 1) ---
    K_VALUES = [3, 5, 10, 20, 40]

    # Data Split
    TEST_SIZE = 0.5  # 50% Valid, 50% Test
    SEED = 42

class ResultLogger:
    def __init__(self, filepath, k_values):
        self.filepath = filepath
        self.k_values = k_values

        # Dynamically create headers based on K_VALUES
        self.metric_cols = []
        for k in self.k_values:
            for m in ["P", "R", "F1"]:
                self.metric_cols.append(f"{m}@{k}")

        self.headers = ["Timestamp", "Model_Name", "Pooling", "Max_Len", "Split"] + \
                       self.metric_cols + ["MRR", "NDCG"]

        # Initialize CSV with headers if it doesn't exist
        if not os.path.exists(filepath):
            with open(filepath, mode='w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(self.headers)

    def log(self, model_name, config_details, split_name, metrics):
        row = [
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            model_name,
            config_details.get("pooling", "N/A"),
            config_details.get("max_length", "N/A"),
            split_name
        ]

        # Add dynamic metrics (P, R, F1)
        for col in self.metric_cols:
            # Map P@3 -> Precision@3 for lookup in metrics dict
            metric_type = col[0] # P, R, or F
            k_val = col.split('@')[1]

            lookup_key = ""
            if metric_type == 'P': lookup_key = f"Precision@{k_val}"
            elif metric_type == 'R': lookup_key = f"Recall@{k_val}"
            elif metric_type == 'F': lookup_key = f"F1@{k_val}"

            row.append(f"{metrics.get(lookup_key, 0):.4f}")

        # Add Fixed metrics
        row.append(f"{metrics.get('MRR', 0):.4f}")
        row.append(f"{metrics.get('NDCG', 0):.4f}")

        with open(self.filepath, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(row)
        print(f"Logged results for {model_name} on {split_name} to {self.filepath}")

logger = ResultLogger(Config.LOG_FILE, Config.K_VALUES)
print("Configuration loaded and Dynamic Logger initialized.")

Configuration loaded and Dynamic Logger initialized.


In [5]:
print("test")
res = faiss.StandardGpuResources()

test


In [6]:
# @title 3. Data Loading, Cleaning & Optimized Filtering
def parse_array_string(x):
    """Parses string representation of numpy arrays or lists from parquet."""
    if isinstance(x, np.ndarray): return x
    if isinstance(x, list): return np.array(x)
    try:
        cleaned = x.replace('[', '').replace(']', '').replace('\n', ' ')
        return np.array([k for k in cleaned.split() if k])
    except:
        return np.array([])

def load_and_prep_data():
    print("Loading Dataframes...")
    df_products = pd.read_parquet(Config.PRODUCT_PATH)
    df_queries = pd.read_parquet(Config.QUERY_PATH)

    print(f"Total Products in DB: {len(df_products)}")
    print(f"Total Queries in Logs: {len(df_queries)}")

    # 1. Parse 'viewed_base_products' and 'result_base_products'
    # We parse lists immediately to enable filtering logic
    tqdm.pandas(desc="Parsing arrays")
    df_queries['viewed_base_products'] = df_queries['viewed_base_products'].progress_apply(parse_array_string)
    df_queries['result_base_products'] = df_queries['result_base_products'].progress_apply(parse_array_string)

    # 2. Filter Empty Queries Immediately
    # We can't use queries with no clicks for valid/test ground truth
    df_queries = df_queries[df_queries['viewed_base_products'].map(len) > 0].copy()
    print(f"Valid Queries (with clicks): {len(df_queries)}")

    # 3. Impute Category ID (Required for Stratified Downsampling)
    print("Imputing Category IDs...")
    # Map all products first (we need this map for imputation before filtering products)
    prod_cat_map = dict(zip(df_products['random_key'], df_products['category_id']))

    def get_imputed_category(row):
        if row['category_id'] != 0: return row['category_id']
        # Find category of clicked products
        clicked_cats = [prod_cat_map.get(k) for k in row['viewed_base_products'] if prod_cat_map.get(k) is not None]
        if not clicked_cats: return 0
        # Mode
        unique, counts = np.unique(clicked_cats, return_counts=True)
        return np.random.choice(unique[counts == counts.max()])

    df_queries['stratify_cat'] = df_queries.apply(get_imputed_category, axis=1)

    # Consolidate rare categories
    cat_counts = df_queries['stratify_cat'].value_counts()
    valid_cats = cat_counts[cat_counts >= 2].index
    df_queries.loc[~df_queries['stratify_cat'].isin(valid_cats), 'stratify_cat'] = -1

    # --- 4. Subsample Queries (Stratified) ---
    # We do this BEFORE filtering products to ensure we only keep products relevant to this subset
    if Config.QUERY_SUBSET_RATIO < 1.0:
        print(f"Downsampling to {Config.QUERY_SUBSET_RATIO*100}% of queries...")
        try:
            df_subset, _ = train_test_split(
                df_queries,
                train_size=Config.QUERY_SUBSET_RATIO,
                stratify=df_queries['stratify_cat'],
                random_state=Config.SEED
            )
            df_queries = df_subset.copy()
        except Exception as e:
            print(f"Stratified sampling failed ({e}). using random.")
            df_queries = df_queries.sample(frac=Config.QUERY_SUBSET_RATIO, random_state=Config.SEED)

        print(f"Queries reduced to: {len(df_queries)}")

        # Re-calculate stratification column for the next Val/Test split
        # (Rare categories might have changed distribution)
        cat_counts = df_queries['stratify_cat'].value_counts()
        valid_cats = cat_counts[cat_counts >= 2].index
        df_queries.loc[~df_queries['stratify_cat'].isin(valid_cats), 'stratify_cat'] = -1

    # --- 5. Filter Products based on the SUBSET of Queries ---
    print("Identifying relevant products for this query subset...")
    # Using itertools chain is often faster for flattening
    import itertools
    valid_keys = set(itertools.chain.from_iterable(df_queries['viewed_base_products']))
    valid_keys.update(itertools.chain.from_iterable(df_queries['result_base_products']))

    initial_p_count = len(df_products)
    df_products = df_products[df_products['random_key'].isin(valid_keys)].copy()
    print(f"Products Filtered: {initial_p_count} -> {len(df_products)}")

    # Garbage collection to free RAM from dropped data
    gc.collect()

    # 6. Split Data (Validation / Test)
    print("Splitting Validation and Test sets...")
    df_val, df_test = train_test_split(
        df_queries,
        test_size=Config.TEST_SIZE,
        stratify=df_queries['stratify_cat'],
        random_state=Config.SEED
    )

    print(f"Validation Set: {len(df_val)}")
    print(f"Test Set: {len(df_test)}")

    return df_products, df_val, df_test

df_products, df_val, df_test = load_and_prep_data()

# Helper: Create text corpus for products
product_texts = df_products['cleaned_text'].tolist()
product_ids = df_products['random_key'].tolist()
id_to_idx = {uid: i for i, uid in enumerate(product_ids)}

Loading Dataframes...
Total Products in DB: 1022297
Total Queries in Logs: 96536


Parsing arrays:   0%|          | 0/96536 [00:00<?, ?it/s]

Parsing arrays:   0%|          | 0/96536 [00:00<?, ?it/s]

Valid Queries (with clicks): 96536
Imputing Category IDs...
Downsampling to 50.0% of queries...
Queries reduced to: 48268
Identifying relevant products for this query subset...
Products Filtered: 1022297 -> 623636
Splitting Validation and Test sets...
Validation Set: 24134
Test Set: 24134


In [7]:
print("test")
res = faiss.StandardGpuResources()

test


In [8]:
# @title 4. Model Architectures & Wrappers
from tqdm.auto import tqdm  # Added for progress bars

# --- A. Reconstructing Custom Architecture (From PDF) ---
class EncoderWithHeads(nn.Module):
    def __init__(self, base_model_path, num_classes, embedding_dim=128, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(base_model_path)
        hidden_size = self.bert.config.hidden_size

        # Contrastive Head
        self.contrastive_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, embedding_dim)
        )
        # Classification Head (Not used for inference but needed for loading weights)
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        emb = self.contrastive_head(cls_token)
        emb = F.normalize(emb, p=2, dim=1)
        return emb

class RetrievalSystem(nn.Module):
    def __init__(self, base_model_path, num_classes, embedding_dim=128):
        super().__init__()
        # Two distinct encoders initialized from same weights
        self.query_encoder = EncoderWithHeads(base_model_path, num_classes, embedding_dim)
        self.product_encoder = EncoderWithHeads(base_model_path, num_classes, embedding_dim)

    def encode_query(self, input_ids, attention_mask):
        return self.query_encoder(input_ids, attention_mask)

    def encode_product(self, input_ids, attention_mask):
        return self.product_encoder(input_ids, attention_mask)

# --- B. Unified Model Wrapper ---
class RetrievalModelWrapper:
    def __init__(self, model_type, model_path_or_obj, tokenizer=None, pooling='cls', max_length=64):
        self.model_type = model_type
        self.model = model_path_or_obj
        self.tokenizer = tokenizer
        self.pooling = pooling
        self.max_length = max_length
        self.device = Config.DEVICE

        if self.model_type != 'sentence_transformer':
            self.model.to(self.device)
            self.model.eval()

    def _tokenize(self, texts):
        return self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        ).to(self.device)

    def _bert_forward(self, texts):
        inputs = self._tokenize(texts)
        with torch.no_grad():
            outputs = self.model(**inputs)

        if self.pooling == 'cls':
            # CLS token is usually index 0
            embeddings = outputs.last_hidden_state[:, 0, :]
        elif self.pooling == 'mean':
            # Mean pooling with attention mask
            last_hidden = outputs.last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = torch.sum(last_hidden * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            embeddings = sum_embeddings / sum_mask

        return F.normalize(embeddings, p=2, dim=1).cpu().numpy()

    def encode(self, texts, mode='query'):
        # mode is used only for dual encoder
        if self.model_type == 'sentence_transformer':
            # Added show_progress_bar=True
            return self.model.encode(texts, batch_size=Config.BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)

        elif self.model_type == 'hf_automodel':
            # Process in batches
            all_embs = []
            # Added tqdm
            for i in tqdm(range(0, len(texts), Config.BATCH_SIZE), desc="Encoding (HF)"):
                batch = texts[i : i + Config.BATCH_SIZE]
                embs = self._bert_forward(batch)
                all_embs.append(embs)
            return np.vstack(all_embs)

        elif self.model_type == 'custom_dual':
            all_embs = []
            # Added tqdm
            for i in tqdm(range(0, len(texts), Config.BATCH_SIZE), desc=f"Encoding ({mode})"):
                batch_texts = texts[i : i + Config.BATCH_SIZE]
                inputs = self._tokenize(batch_texts)
                with torch.no_grad():
                    if mode == 'query':
                        emb = self.model.encode_query(inputs['input_ids'], inputs['attention_mask'])
                    else:
                        emb = self.model.encode_product(inputs['input_ids'], inputs['attention_mask'])
                all_embs.append(emb.cpu().numpy())
            return np.vstack(all_embs)

In [9]:
print("test")
res = faiss.StandardGpuResources()

test


In [10]:
# @title 5. Evaluation Engine (GPU/CPU Toggable)
class Evaluator:
    def __init__(self, df_data, product_ids, id_to_idx):
        self.df = df_data
        self.product_ids = product_ids
        self.id_to_idx = id_to_idx
        print("test")
        res = faiss.StandardGpuResources()

    def compute_metrics(self, retrieved_indices, k_values):
        # Initialize dictionary for ALL k_values
        metrics = {f"{m}@{k}": 0.0 for k in k_values for m in ["Precision", "Recall", "F1"]}
        metrics.update({"MRR": 0.0, "NDCG": 0.0})

        total_queries = 0

        for i, row in tqdm(enumerate(self.df.itertuples()), total=len(self.df), desc="Calc Metrics"):
            true_keys = row.viewed_base_products
            if len(true_keys) == 0: continue

            true_indices = {self.id_to_idx[k] for k in true_keys if k in self.id_to_idx}
            if not true_indices: continue

            total_queries += 1
            pred_indices = retrieved_indices[i]

            # MRR
            first_hit = -1
            for rank, idx in enumerate(pred_indices):
                if idx in true_indices:
                    first_hit = rank + 1
                    break
            if first_hit != -1: metrics["MRR"] += 1.0 / first_hit

            # NDCG
            dcg, idcg = 0.0, 0.0
            for j in range(min(len(true_indices), max(k_values))):
                idcg += 1.0 / np.log2(j + 2)
            for j, idx in enumerate(pred_indices):
                if j >= max(k_values): break
                rel = 1 if idx in true_indices else 0
                dcg += rel / np.log2(j + 2)
            if idcg > 0: metrics["NDCG"] += dcg / idcg

            # Metrics @ K
            for k in k_values:
                top_k = set(pred_indices[:k])
                hits = len(top_k & true_indices)
                prec = hits / k
                rec = hits / len(true_indices)
                f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
                metrics[f"Precision@{k}"] += prec
                metrics[f"Recall@{k}"] += rec
                metrics[f"F1@{k}"] += f1

        for k in metrics: metrics[k] /= total_queries
        return metrics

    def run(self, wrapper, k_values=Config.K_VALUES):
        print("test")
        res = faiss.StandardGpuResources()
        # 1. Build CPU Index first (RAM Safe)
        dummy_emb = wrapper.encode(["test"], mode='product')
        d = dummy_emb.shape[1]
        cpu_index = faiss.IndexFlatIP(d)

        print("Indexing Products (CPU)...")
        index_batch_size = 10000
        for i in tqdm(range(0, len(product_texts), index_batch_size), desc="Indexing"):
            batch_texts = product_texts[i : i + index_batch_size]
            batch_embs = wrapper.encode(batch_texts, mode='product')
            cpu_index.add(batch_embs)
            del batch_embs
            gc.collect()

        print(f"Total Products Indexed: {cpu_index.ntotal}")

        # 2. Setup Search Index (GPU vs CPU)
        if Config.USE_FAISS_GPU:
            print("Moving Index to GPU for Fast Search...")
            try:
                res = faiss.StandardGpuResources()
                search_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
            except Exception as e:
                print(f"GPU Indexing failed ({e}). Fallback to CPU.")
                search_index = cpu_index
        else:
            print("Using CPU Index (Config.USE_FAISS_GPU=False)...")
            search_index = cpu_index

        # 3. Search
        print("Encoding Queries...")
        query_texts = self.df['cleaned_query'].tolist()
        query_embs = wrapper.encode(query_texts, mode='query')

        print(f"Retrieving top {max(k_values)}...")
        _, I = search_index.search(query_embs, max(k_values))

        # Cleanup
        if Config.USE_FAISS_GPU and 'search_index' in locals():
            del search_index
            if 'res' in locals(): del res
            torch.cuda.empty_cache()

        return self.compute_metrics(I, k_values)

In [11]:
# @title 6. Main Pipeline Execution
from sentence_transformers import SentenceTransformer

def run_pipeline():
    print("test")
    res = faiss.StandardGpuResources()
    experiments = []

    # Define Experiments
    experiments.append({
        "name": "Heydari-Persian-Emb", "type": "sentence_transformer",
        "path": Config.HEYDARI_MODEL, "poolings": ["default"], "lengths": [64]
    })
    experiments.append({
        "name": "Hooshvare-DistilBert", "type": "hf_automodel",
        "path": Config.HOOSHVARE_BASE, "poolings": ["cls"], "lengths": [64]
    })
    experiments.append({
        "name": "Custom-MLM-Base", "type": "hf_automodel",
        "path": Config.CUSTOM_BASE_PATH, "poolings": ["cls"], "lengths": [64]
    })
    experiments.append({
        "name": "Custom-Dual-Contrastive", "type": "custom_dual",
        "path": Config.CUSTOM_FULL_PTH, "base_path": Config.CUSTOM_BASE_PATH,
        "poolings": ["cls"], "lengths": [64]
    })

    best_score, best_config, best_wrapper = -1, None, None
    print("=== Starting Validation Phase ===")

    for exp in experiments:
        for pooling in exp["poolings"]:
            for length in exp["lengths"]:
                print(f"\n--- Running: {exp['name']} | Pool: {pooling} | Len: {length} ---")
                try:
                    wrapper = None
                    if exp["type"] == "sentence_transformer":
                        model = SentenceTransformer(exp["path"])
                        wrapper = RetrievalModelWrapper("sentence_transformer", model, max_length=length)
                    elif exp["type"] == "hf_automodel":
                        tokenizer = AutoTokenizer.from_pretrained(exp["path"])
                        model = AutoModel.from_pretrained(exp["path"])
                        wrapper = RetrievalModelWrapper("hf_automodel", model, tokenizer, pooling, length)
                    elif exp["type"] == "custom_dual":
                        tokenizer = AutoTokenizer.from_pretrained(exp["base_path"])
                        state_dict = torch.load(exp["path"], map_location='cpu')
                        # Heuristic to find num_classes from weights
                        cls_weight = state_dict.get('product_encoder.classification_head.3.weight')
                        num_classes = cls_weight.shape[0] if cls_weight is not None else 338
                        custom_model = RetrievalSystem(exp["base_path"], num_classes)
                        custom_model.load_state_dict(state_dict)
                        wrapper = RetrievalModelWrapper("custom_dual", custom_model, tokenizer, pooling, length)

                    evaluator = Evaluator(df_val, product_ids, id_to_idx)
                    metrics = evaluator.run(wrapper)

                    logger.log(exp["name"], {"pooling": pooling, "max_length": length}, "Validation", metrics)

                    if metrics["NDCG"] > best_score:
                        best_score = metrics["NDCG"]
                        best_config = {"name": exp["name"], "details": {"pooling": pooling, "max_length": length}}
                        best_wrapper = wrapper

                    gc.collect(); torch.cuda.empty_cache()
                except Exception as e:
                    print(f"Skipping {exp['name']} error: {e}")

    print(f"\n=== Best Model: {best_config['name']} (NDCG: {best_score:.4f}) ===")

    print("Running Final Test Evaluation...")
    evaluator_test = Evaluator(df_test, product_ids, id_to_idx)
    test_metrics = evaluator_test.run(best_wrapper)

    logger.log(best_config["name"], best_config["details"], "TEST_FINAL", test_metrics)
    for k, v in test_metrics.items(): print(f"{k}: {v:.4f}")

run_pipeline()

test
=== Starting Validation Phase ===

--- Running: Heydari-Persian-Emb | Pool: default | Len: 64 ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

Total Products Indexed: 623636
Moving Index to GPU for Fast Search...
Encoding Queries...


Encoding (query):   0%|          | 0/378 [00:00<?, ?it/s]

Retrieving top 40...


Calc Metrics:   0%|          | 0/24134 [00:00<?, ?it/s]

Logged results for Custom-Dual-Contrastive on TEST_FINAL to /content/drive/MyDrive/RAGSearch/experiment_logs.csv
Precision@3: 0.0478
Recall@3: 0.0909
F1@3: 0.0574
Precision@5: 0.0400
Recall@5: 0.1238
F1@5: 0.0558
Precision@10: 0.0310
Recall@10: 0.1866
F1@10: 0.0499
Precision@20: 0.0229
Recall@20: 0.2694
F1@20: 0.0404
Precision@40: 0.0160
Recall@40: 0.3683
F1@40: 0.0299
MRR: 0.1252
NDCG: 0.1626
